# 5 — Évaluer les outils NLP

Section 5 du livret et annexe B du guide pratique.

## 5.1 Les quatre tableaux à produire

Les 4 tableaux attendus :
1. Extraction : VP/FP/FN, précision, rappel, F1 (micro-moyenne globale + par catégorie), temps.
2. Négation : matrice de confusion, P/R/F1 par classe, moyenne des deux F1 (10 exemples test).
3. Recherche QA : Hit@5, Rappel@5, couverture des éléments (questions répondables uniquement).
4. Réponses QA : BLEU (corpus, 0-100), ROUGE-1/2/L (F1 moyenné par question, 0-1), analyse manuelle (exactitude, complétude, citations, abstentions).

### Tableau 1 — Extraction sur TEST (scoreur commun : `src/extraction.py`)

Versions figées sur VALIDATION, évaluées ici **une seule fois**, sans nouveau réglage :
- méthode par dictionnaire et règles : **DIC-v3** (notebook 2.3b) ;
- méthode apprise : **APP-v3**, PubMedBERT (notebook 2.4c).

Même référence, mêmes conventions (correspondance exacte du document, des bornes et du type), micro-moyenne.

In [1]:
import sys, json
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import erreurs
from src.cdr import charger_jsonl, mentions_reference
from src.extraction import tableau_scores, lire_predictions

EXTRACTION = RACINE / "resultats" / "extraction"
documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")
test = [d for d in documents if d.partition == "test"]
ref_test = mentions_reference(test)
docs_test = {d.document_id for d in test}
print(len(test), "documents TEST ;", len(ref_test), "mentions de référence")

METHODES = {   # nom dans le tableau : (fichier de prédictions, fichier de configuration, clé du temps TEST)
    "dictionnaire (DIC-v3)": ("predictions_dictionnaire_v3_test.jsonl", "dictionnaire_v3_config_gelee.json",
                              ("temps_s", "extraction_test")),
    "apprise (APP-v3, PubMedBERT)": ("predictions_apprise_v3_test.jsonl", "apprise_v3_config_gelee.json",
                                     ("temps_s", "prediction_test")),
}
preds, tableaux = {}, []
for nom, (fichier, config, (a, b)) in METHODES.items():
    preds[nom] = {m for m in lire_predictions(EXTRACTION / fichier) if m.doc in docs_test}
    temps = json.loads((EXTRACTION / config).read_text(encoding="utf-8"))[a][b]
    tableaux.append(tableau_scores(ref_test, preds[nom], nom, "test", temps))
scores_test = pd.concat(tableaux, ignore_index=True)
scores_test.to_csv(EXTRACTION / "scores_test.csv", index=False)
scores_test

500 documents TEST ; 9809 mentions de référence


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,dictionnaire (DIC-v3),test,Chemical,4085,401,1300,0.9106,0.7586,0.8277,False,NaN
1,dictionnaire (DIC-v3),test,Disease,3135,669,1289,0.8241,0.7086,0.7620,False,NaN
2,dictionnaire (DIC-v3),test,global (micro),7220,1070,2589,0.8709,0.7361,0.7978,False,3.32
3,"apprise (APP-v3, PubMedBERT)",test,Chemical,5085,486,300,0.9128,0.9443,0.9283,False,NaN
4,"apprise (APP-v3, PubMedBERT)",test,Disease,3843,749,581,0.8369,0.8687,0.8525,False,NaN
5,"apprise (APP-v3, PubMedBERT)",test,global (micro),8928,1235,881,0.8785,0.9102,0.8941,False,16.80


In [2]:
# Rappel des scores VALIDATION des mêmes versions, pour voir l'écart VALIDATION → TEST.
val = pd.concat([pd.read_csv(EXTRACTION / "scores_dictionnaire_v3_validation.csv").assign(methode="dictionnaire (DIC-v3)"),
                 pd.read_csv(EXTRACTION / "scores_apprise_v3_validation.csv").assign(methode="apprise (APP-v3, PubMedBERT)")])
comparaison = (val.set_index(["methode", "categorie"])["F1"].rename("F1 VALIDATION").to_frame()
               .join(scores_test.set_index(["methode", "categorie"])["F1"].rename("F1 TEST")))
comparaison["écart"] = (comparaison["F1 TEST"] - comparaison["F1 VALIDATION"]).round(4)
comparaison

F1 VALIDATION  F1 TEST   écart
methode                      categorie                                     
dictionnaire (DIC-v3)        Chemical               0.8333   0.8277 -0.0056
                             Disease                0.7569   0.7620  0.0051
                             global (micro)         0.7993   0.7978 -0.0015
apprise (APP-v3, PubMedBERT) Chemical               0.9532   0.9283 -0.0249
                             Disease                0.8598   0.8525 -0.0073
                             global (micro)         0.9119   0.8941 -0.0178

In [3]:
# Types d'erreurs sur TEST (mêmes catégories qu'au 2.3 / 2.4)
pd.DataFrame({nom: {k: len(v) for k, v in erreurs(ref_test, p).items()} for nom, p in preds.items()}).T

,correcte,frontiere,type,manquee,parasite
dictionnaire (DIC-v3),7220,695,7,1887,368
"apprise (APP-v3, PubMedBERT)",8928,476,3,402,756


In [4]:
# Sorties des deux méthodes sur TEST, au format du guide (data/predictions_extraction.jsonl)
with open(RACINE / "data" / "predictions_extraction.jsonl", "w", encoding="utf-8") as sortie:
    for nom, (fichier, _, _) in METHODES.items():
        methode = "dictionnaire" if nom.startswith("dictionnaire") else "apprise"
        for ligne in (EXTRACTION / fichier).read_text(encoding="utf-8").splitlines():
            o = json.loads(ligne)
            o["methode"] = methode
            sortie.write(json.dumps(o, ensure_ascii=False) + "\n")
print(sum(1 for _ in open(RACINE / "data" / "predictions_extraction.jsonl")), "lignes écrites")

2000 lignes écrites


### Tableau 2 — Négation : matrice de confusion (lignes = référence, colonnes = prédit)

Résultats du notebook 2.5 (règles NEG-v0 sur les 10 phrases de test). Référence **finale** : double annotation
(`annotation_AMARA.csv`, `annotation_TOMAS.csv`, kappa de Cohen 0,51), désaccords résolus ensemble dans
`desaccords_annotation.csv`.

In [5]:
NEGATION = RACINE / "resultats" / "negation"
display(pd.read_csv(NEGATION / "matrice_confusion.csv", index_col=0))
pd.read_csv(NEGATION / "scores_par_classe.csv")

,nie,non_nie
référence,,
nie,4,1
non_nie,1,4


,reference_provisoire,classe,VP,FP,FN,precision,rappel,F1,support,denominateur_nul
0,False,nie,4.0,1.0,1.0,0.8,0.8,0.8,5.0,False
1,False,non_nie,4.0,1.0,1.0,0.8,0.8,0.8,5.0,False
2,False,moyenne des deux F1,NaN,NaN,NaN,NaN,NaN,0.8,NaN,NaN


### Tableau 3 — Recherche QA sur TEST : Hit@5, Rappel@5, couverture des éléments (questions avec réponse)

Configuration figée sur DEV (`resultats/qa/qa_config_gelee.json`, notebook 4.2), interrogée ici **une seule fois** :
- **socle** : BM25 REC-v2 (texte, racinisation Snowball, k1 = 1,5, b = 0,75), 5 passages ;
- **approfondissement** : recherche dense E5 sur titre de section + texte (notebook 4.3). DEV a servi à la choisir :
  son avantage sur DEV est optimiste, TEST dit s'il se confirme.

In [6]:
import time
from src.qa_recherche import construire_bm25, scores_bm25, premiers, mesures_question, IndexDense, K1, B
from src.qa_llm import ClientLLM, formater_passages, diagnostiquer
from src.qa_scores import scores_reponses

DATA, QA = RACINE / "data", RACINE / "resultats" / "qa"
gel = json.loads((QA / "qa_config_gelee.json").read_text(encoding="utf-8"))
K = gel["recherche"]["k"]
passages = [json.loads(l) for l in (DATA / "passages.jsonl").read_text(encoding="utf-8").splitlines()]
par_id = {p["passage_id"]: p for p in passages}
ids, textes = [p["passage_id"] for p in passages], [p["texte"] for p in passages]
questions = [json.loads(l) for l in (DATA / "questions_test.jsonl").read_text(encoding="utf-8").splitlines()]
questions_par_id = {q["question_id"]: q for q in questions}
print(len(questions), "questions TEST dont", sum(q["answerable"] for q in questions), "avec réponse ; gel du", gel["date_gel"])

bm25 = construire_bm25(textes, racines=True)
dense = IndexDense([f'{p["localisation"]["section"]} {t}' for p, t in zip(passages, textes)])
RECHERCHES = {"BM25 REC-v2 (socle)": (gel["recherche"]["socle"], lambda q: scores_bm25(q, bm25, K1, B)),
              "dense (approfondissement)": (gel["recherche"]["approfondissement"], dense.scores)}

classements = {nom: {} for nom in RECHERCHES}   # nom → question_id → (top-5, durée)
for nom, (_, scorer) in RECHERCHES.items():
    for q in questions:
        debut = time.perf_counter()
        top = [{"passage_id": ids[i], "rang": r, "score": round(s, 4)}
               for r, (i, s) in enumerate(premiers(scorer(q["question"]), K), start=1)]
        classements[nom][q["question_id"]] = (top, time.perf_counter() - debut)

detail_recherche = pd.DataFrame([{"recherche": nom, "question_id": q["question_id"],
                                  **mesures_question(q, [p["passage_id"] for p in c[q["question_id"]][0]], K)}
                                 for nom, c in classements.items() for q in questions if q["answerable"]])
detail_recherche.to_csv(QA / "recherche_test.csv", index=False)
MESURES = [f"hit@{K}", f"rappel@{K}", f"couverture_elements@{K}"]
recherche_test = detail_recherche.groupby("recherche", sort=False)[MESURES].mean().round(3)
recherche_test

10 questions TEST dont 8 avec réponse ; gel du 2026-10-30


/home/pico-dev/venvs/nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,hit@5,rappel@5,couverture_elements@5
recherche,,,
BM25 REC-v2 (socle),0.875,0.771,0.792
dense (approfondissement),1.000,0.771,0.792


In [7]:
# Rappel des scores DEV (notebook 4.3) et rang de la 1re preuve par question TEST.
dev_recherche = (pd.read_csv(QA / "recherche_dev_hybride.csv", index_col=0)
                 .loc[["BM25 REC-v2", "dense (section + texte)"], MESURES].set_axis(list(RECHERCHES)))
display(pd.concat({"DEV": dev_recherche, "TEST": recherche_test}, axis=1))
detail_recherche.pivot(index="recherche", columns="question_id", values="rang_premiere_preuve")

DEV                                  TEST  \
                           hit@5 rappel@5 couverture_elements@5  hit@5   
BM25 REC-v2 (socle)        0.875    0.781                 0.823  0.875   
dense (approfondissement)  0.938    0.875                 0.979  1.000   

                                                          
                          rappel@5 couverture_elements@5  
BM25 REC-v2 (socle)          0.771                 0.792  
dense (approfondissement)    0.771                 0.792

question_id,Q021,Q022,Q023,Q024,Q025,Q026,Q027,Q028
recherche,,,,,,,,
BM25 REC-v2 (socle),2.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0
dense (approfondissement),1.0,2.0,1.0,1.0,4.0,2.0,1.0,1.0


### Tableau 4 — Réponses QA sur TEST : BLEU / ROUGE (questions avec réponse)

Génération unique avec la configuration figée : Q0 = premier passage BM25 tel quel ; Q1 v1 et Q2 v1, contexte c2,
`gpt-4.1-mini-2025-04-14`, température 0, graine 42, 300 tokens, mêmes 5 passages BM25 ; Q2 avec les passages de
la recherche dense en approfondissement. Les requêtes passent par le cache (`cache_llm.json`) : une réexécution
redonne les mêmes réponses sans nouvel appel. **En cas d'échec d'un appel (clé absente, réseau), rien n'est écrit**
et on relance une fois le problème réglé : un échec n'est pas un résultat.

Sorties : `data/predictions_qa.jsonl` (Q0, Q1, Q2 : les 30 réponses du livret) et
`resultats/qa/predictions_llm_test_dense.jsonl` (approfondissement). Calcul BLEU/ROUGE : `src/qa_scores.py`, mêmes
conventions que sur DEV (notebook 4.2, annexe B.4 du guide).

In [8]:
p = gel["llm"]
llm = ClientLLM(p["modele"], p["temperature"], p["seed"], p["max_tokens"], QA / "cache_llm.json")
catalogue = pd.read_csv(DATA / "corpus.csv", dtype=str)
TITRES = dict(zip(catalogue.document_id, catalogue.titre.str.replace(
    r" \((page web|guide patients[^)]*|fiche Cancer info)\)$", "", regex=True)))   # comme dans le notebook 4.2

def q0(q):
    top, duree = classements["BM25 REC-v2 (socle)"][q["question_id"]]
    premier = top[0]["passage_id"]
    return {"configuration": "Q0", "question_id": q["question_id"], "partition": "test",
            "recherche": gel["recherche"]["socle"], "passages_retrouves": top, "passages_transmis": [premier],
            "reponse": f'{par_id[premier]["texte"]} [{premier}]', "duree_secondes": round(duree, 4), "echec": None}

def llm_reponse(q, config, recherche):
    c = gel["configurations"][config]
    top = classements[recherche][q["question_id"]][0]
    transmis = [par_id[t["passage_id"]] for t in top]
    prompt = c["prompt"].format(question=q["question"], passages=formater_passages(transmis, TITRES))
    s = llm.generer(prompt)
    return {"configuration": config, "question_id": q["question_id"], "partition": "test",
            "recherche": RECHERCHES[recherche][0], "passages_retrouves": top,
            "passages_transmis": [t["passage_id"] for t in top], "prompt_version": c["prompt_version"],
            "contexte_version": gel["contexte"], "prompt": prompt, "reponse": s["texte"],
            "duree_secondes": s.get("duree_secondes"), "echec": s["echec"], "modele": p["modele"],
            "parametres": {k: p[k] for k in ("temperature", "seed", "max_tokens")},
            **{k: s.get(k) for k in ("modele_reponse", "system_fingerprint", "fin", "tokens_prompt", "tokens_reponse",
                                     "origine")}}

reponses = {"Q0": [q0(q) for q in questions],
            "Q1": [llm_reponse(q, "Q1", "BM25 REC-v2 (socle)") for q in questions],
            "Q2": [llm_reponse(q, "Q2", "BM25 REC-v2 (socle)") for q in questions],
            "Q2_dense": [llm_reponse(q, "Q2_dense", "dense (approfondissement)") for q in questions]}
echecs = [(c, l["question_id"], l["echec"]) for c, ls in reponses.items() for l in ls if l["echec"]]
if echecs:
    raise RuntimeError(f"{len(echecs)} appels en échec, rien n'est écrit ; premier : {echecs[0]}")

def ecrire(chemin, lignes):
    with open(chemin, "w", encoding="utf-8") as f:
        for l in lignes:
            f.write(json.dumps(l, ensure_ascii=False) + "\n")
ecrire(DATA / "predictions_qa.jsonl", reponses["Q0"] + reponses["Q1"] + reponses["Q2"])
ecrire(QA / "predictions_llm_test_dense.jsonl", reponses["Q2_dense"])
for c, ls in reponses.items():
    print(f"{c} : {len(ls)} réponses, tokens {sum(l.get('tokens_prompt') or 0 for l in ls)} + "
          f"{sum(l.get('tokens_reponse') or 0 for l in ls)}, origine {pd.Series([l.get('origine', '-') for l in ls]).value_counts().to_dict()}")

Q0 : 10 réponses, tokens 0 + 0, origine {'-': 10}
Q1 : 10 réponses, tokens 8441 + 891, origine {'api': 10}
Q2 : 10 réponses, tokens 9281 + 810, origine {'api': 10}
Q2_dense : 10 réponses, tokens 8334 + 796, origine {'api': 10}


In [9]:
NOMS = {"Q0": "Q0 (BM25)", "Q1": "Q1 v1 c2 (BM25)", "Q2": "Q2 v1 c2 (BM25)", "Q2_dense": "Q2 v1 c2 (dense, approfondissement)"}
bleu_rouge = pd.DataFrame({NOMS[c]: scores_reponses(ls, questions_par_id) for c, ls in reponses.items()}).T
print("BLEU :", bleu_rouge.pop("signature BLEU").iloc[0], "; ROUGE : rouge-score 0.1.2, tokeniseur commun \\w+")
bleu_rouge.to_csv(QA / "bleu_rouge.csv")
bleu_rouge

BLEU : nrefs:1|case:mixed|eff:no|tok:none|smooth:exp|version:2.6.0 ; ROUGE : rouge-score 0.1.2, tokeniseur commun \w+


,questions,BLEU,ROUGE-1,ROUGE-2,ROUGE-L,réponses vides,mots (médiane)
Q0 (BM25),8,5.44,0.297,0.102,0.21,0,105.5
Q1 v1 c2 (BM25),8,9.96,0.422,0.173,0.315,0,55.0
Q2 v1 c2 (BM25),8,10.1,0.43,0.178,0.325,0,59.0
"Q2 v1 c2 (dense, approfondissement)",8,13.99,0.455,0.222,0.354,0,63.0


In [10]:
# Contrôles automatiques (aide à la lecture, pas la grille manuelle) : abstentions, citations, longueur.
diag = pd.DataFrame([{"configuration": NOMS[c], "avec_reponse": questions_par_id[l["question_id"]]["answerable"],
                      **diagnostiquer(l["reponse"], l["passages_transmis"])}
                     for c, ls in reponses.items() if c != "Q0" for l in ls])
n_sans = sum(not q["answerable"] for q in questions)
diag.groupby("configuration", sort=False).apply(lambda d: pd.Series({
    f"abstentions correctes (/{n_sans})": int((d.abstention & ~d.avec_reponse).sum()),
    f"phrase d'abstention sur questions avec réponse (/{len(questions) - n_sans})": int((d.abstention & d.avec_reponse).sum()),
    "citations inventées": int(d.citations_inventees.map(len).sum()),
    "mots (médiane)": d.mots.median()}), include_groups=False)

,abstentions correctes (/2),phrase d'abstention sur questions avec réponse (/8),citations inventées,mots (médiane)
configuration,,,,
Q1 v1 c2 (BM25),0.0,0.0,0.0,53.5
Q2 v1 c2 (BM25),1.0,0.0,0.0,53.0
"Q2 v1 c2 (dense, approfondissement)",2.0,1.0,0.0,55.5


### Comparaison avec un second LLM : Mistral (TEST)

Les mêmes prompts TEST (Q1, Q2, Q2 dense : mêmes passages, mêmes consignes) ont été envoyés à `mistral-medium-2604`
(température 0, graine 42, 300 tokens) dans `4.4_comparaison_mistral.ipynb` ; seul le LLM change, la configuration
figée n'est pas modifiée. On relit ici ses réponses (`resultats/qa/predictions_mistral.jsonl`), sans nouvel appel.

In [ ]:
MISTRAL = QA / "predictions_mistral.jsonl"
CONFIG_MISTRAL = {"TEST Q1 v1 c2": "Q1", "TEST Q2 v1 c2": "Q2", "TEST Q2 v1 c2, dense": "Q2_dense"}
if not MISTRAL.exists():
    print("Réponses Mistral absentes : exécuter notebooks/4.4_comparaison_mistral.ipynb, puis cette cellule.")
else:
    mistral = {c: [] for c in CONFIG_MISTRAL.values()}
    for l in map(json.loads, MISTRAL.read_text(encoding="utf-8").splitlines()):
        if l["experience"] in CONFIG_MISTRAL:
            mistral[CONFIG_MISTRAL[l["experience"]]].append(l)
    assert all([l["prompt"] for l in mistral[c]] == [l["prompt"] for l in reponses[c]] for c in mistral)  # mêmes prompts

    def resume(c, llm, lignes):
        d = pd.DataFrame([{"avec_reponse": questions_par_id[l["question_id"]]["answerable"],
                           **diagnostiquer(l["reponse"], l["passages_transmis"])} for l in lignes])
        s = scores_reponses(lignes, questions_par_id)
        return {"configuration": NOMS[c], "LLM": llm, **{k: s[k] for k in ("BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L")},
                f"abstentions correctes (/{n_sans})": int((d.abstention & ~d.avec_reponse).sum()),
                f"phrase d'abstention sur questions avec réponse (/{len(questions) - n_sans})": int((d.abstention & d.avec_reponse).sum()),
                "citations inventées": int(d.citations_inventees.map(len).sum()), "mots (médiane)": s["mots (médiane)"]}

    openai_mistral = pd.DataFrame([resume(c, llm, ls) for c in mistral
                                   for llm, ls in (("gpt-4.1-mini", reponses[c]), ("mistral-medium", mistral[c]))])
    display(openai_mistral.set_index(["configuration", "LLM"]))

BLEU  ROUGE-1  ROUGE-2  \
configuration                       LLM                                       
Q1 v1 c2 (BM25)                     gpt-4.1-mini     9.96    0.422    0.173   
                                    mistral-medium  11.61    0.451    0.193   
Q2 v1 c2 (BM25)                     gpt-4.1-mini    10.10    0.430    0.178   
                                    mistral-medium  11.66    0.463    0.191   
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini    13.99    0.455    0.222   
                                    mistral-medium  15.34    0.477    0.225   

                                                    ROUGE-L  \
configuration                       LLM                       
Q1 v1 c2 (BM25)                     gpt-4.1-mini      0.315   
                                    mistral-medium    0.343   
Q2 v1 c2 (BM25)                     gpt-4.1-mini      0.325   
                                    mistral-medium    0.359   
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini      0.354   
                                    mistral-medium    0.380   

                                                    abstentions correctes (/2)  \
configuration                       LLM                                          
Q1 v1 c2 (BM25)                     gpt-4.1-mini                             0   
                                    mistral-medium                           0   
Q2 v1 c2 (BM25)                     gpt-4.1-mini                             1   
                                    mistral-medium                           2   
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini                             2   
                                    mistral-medium                           2   

                                                    phrase d'abstention sur questions avec réponse (/8)  \
configuration                       LLM                                                                   
Q1 v1 c2 (BM25)                     gpt-4.1-mini                                                    0     
                                    mistral-medium                                                  0     
Q2 v1 c2 (BM25)                     gpt-4.1-mini                                                    0     
                                    mistral-medium                                                  0     
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini                                                    1     
                                    mistral-medium                                                  1     

                                                    citations inventées  \
configuration                       LLM                                   
Q1 v1 c2 (BM25)                     gpt-4.1-mini                      0   
                                    mistral-medium                    0   
Q2 v1 c2 (BM25)                     gpt-4.1-mini                      0   
                                    mistral-medium                    0   
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini                      0   
                                    mistral-medium                    0   

                                                    mots (médiane)  
configuration                       LLM                             
Q1 v1 c2 (BM25)                     gpt-4.1-mini              55.0  
                                    mistral-medium            48.5  
Q2 v1 c2 (BM25)                     gpt-4.1-mini              59.0  
                                    mistral-medium            39.5  
Q2 v1 c2 (dense, approfondissement) gpt-4.1-mini              63.0  
                                    mistral-medium            45.0

**Lecture** : avec les mêmes passages, Mistral obtient des BLEU / ROUGE un peu meilleurs, avec des réponses plus
courtes, et s'abstient correctement sur les deux questions sans réponse en Q2 (Q030 : OpenAI répondait sans la phrase
imposée). Les conclusions du tableau 4 ne changent pas : les deux LLM se trompent aux mêmes endroits, surtout quand la
recherche ne transmet pas la preuve (Q025 avec BM25) ; aucun ne signale la divergence de sources de Q022. Mistral ne
fait pas la généralisation « grippe » de Q024, mais ajoute à Q025 « l'anxiété », absente du passage transmis. Détail
et comparaison sur DEV : notebook 4.4 et `RAPPORT.md` (LLM-MIS-v1).

## 5.3 Examiner les réponses et expliquer les erreurs

### Grille de lecture (30 réponses, jugées par un juge LLM indépendant)

Guide, annexe B.5. Les 30 réponses (10 questions × Q0/Q1/Q2) sont mélangées (graine 42) et présentées sans nom de
méthode dans `resultats/qa/grille_manuelle.csv` ; la correspondance réponse → configuration est à part
(`grille_correspondance.csv`, à n'ouvrir qu'après les jugements). Les colonnes `_juge` ont été remplies par un juge LLM
(Claude) sans aucun contexte du projet : il ne voyait que la question, la référence, les éléments attendus, le texte des
passages transmis et la réponse. Deux juges lancés séparément ont donné exactement les mêmes notes (30/30) ; un seul
jeu est gardé. Le fichier n'est créé qu'une fois (les jugements ne sont jamais écrasés).
- **exactitude** : 2 aucune affirmation fausse ou non soutenue ; 1 mélange ; 0 réponse centrale fausse ou absente
  (abstention sur une question avec réponse = 0) ;
- **completude** : nombre d'éléments attendus correctement présents (sur `nb_elements`) ;
- **citations** : liens affirmation-citation soutenus / liens proposés ;
- **abstention** : correcte, à tort, ou manquante (réponse donnée à une question sans réponse).

In [ ]:
GRILLE, CORRESP = QA / "grille_manuelle.csv", QA / "grille_correspondance.csv"
if not GRILLE.exists():
    trente = pd.DataFrame([{**l, "configuration": c} for c in ("Q0", "Q1", "Q2") for l in reponses[c]])
    trente = trente.sample(frac=1, random_state=42).reset_index(drop=True)
    trente["id_reponse"] = [f"R{i:02d}" for i in range(1, len(trente) + 1)]
    grille = pd.DataFrame({
        "id_reponse": trente.id_reponse, "question_id": trente.question_id,
        "question": trente.question_id.map(lambda i: questions_par_id[i]["question"]),
        "avec_reponse": trente.question_id.map(lambda i: questions_par_id[i]["answerable"]),
        "reference": trente.question_id.map(lambda i: questions_par_id[i]["reference"]),
        "elements": trente.question_id.map(lambda i: " | ".join(questions_par_id[i]["elements"])),
        "nb_elements": trente.question_id.map(lambda i: len(questions_par_id[i]["elements"])),
        "passages_transmis": trente.passages_transmis.map(", ".join), "reponse": trente.reponse})
    for dim in ("exactitude", "completude", "citations", "abstention"):
        grille[f"{dim}_juge"] = ""
    grille.to_csv(GRILLE, index=False)
    trente[["id_reponse", "configuration", "question_id"]].to_csv(CORRESP, index=False)
    print("Grille créée :", len(grille), "réponses")
else:
    print("Grille existante conservée :", GRILLE.name)

### Analyse des erreurs

Extraction : forme inconnue, mauvaise frontière, confusion de type.
Négation : cas difficiles (coordination, « impossible d'exclure »).
QA : passage manqué, réponse incomplète, affirmation non soutenue, absence d'information, désaccord score automatique / lecture humaine.